In [1]:
import pandas as pd
import os
import json
import numpy as np
from os.path import dirname

root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/


In [2]:
with open("dataset_features.json", 'r') as file:
    datasets_info = json.load(file)


In [3]:
list(datasets_info.keys())

['bpic2015_1',
 'bpic2015_2',
 'bpic2015_3',
 'bpic2015_4',
 'bpic2015_5',
 'credit',
 'helpdesk',
 'invoice',
 'Production_Data',
 'sepsis']

In [4]:
dataset = "bpic2015_5"

In [5]:
tab_all = pd.read_csv(f"datasets/processed/{dataset}_processed_all.csv")
tab_all.head()

/tmp/ipykernel_8666/633048827.py:1: DtypeWarning: Columns (21,23) have mixed types. Specify dtype option on import or set low_memory=False.
  tab_all = pd.read_csv(f"datasets/processed/{dataset}_processed_all.csv")


,Responsible_actor,SUMleges,CaseID,Aanleg (Uitvoeren werk of werkzaamheid),Bouw,Brandveilig gebruik (melding),Brandveilig gebruik (vergunning),Flora en Fauna,Gebiedsbescherming,Handelen in strijd met regels RO,Inrit/Uitweg,Integraal,Kap,Milieu (melding),Milieu (neutraal wijziging),Milieu (omgevingsvergunning beperkte milieutoets),Milieu (vergunning),Monument,Reclame,Sloop,Activity,monitoringResource,question,Resource,time:timestamp,duration,month,weekday,hour,remtime,elapsed,remaining_time
0,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_180\\complete,560600,EMPTY,560429,1.300915e+09,0.016667,3,3,1,0.0,14750458.0,0.0
1,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_150\\complete,560600,EMPTY,560429,1.300915e+09,0.000000,3,3,1,1.0,14750457.0,1.0
2,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_110\\complete,560600,False,560429,1.300915e+09,159380.950000,3,3,1,1.0,14750457.0,1.0
3,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_090_2\\complete,560600,EMPTY,560600,1.291352e+09,528.283333,12,4,9,9562858.0,5187600.0,9562858.0
4,560600,0.0,3364103,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_090_1\\complete,560600,EMPTY,560600,1.291321e+09,100.866667,12,4,0,9594555.0,5155903.0,9594555.0


In [6]:
tab_train = pd.read_csv(f"datasets/processed/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"datasets/processed/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"datasets/processed/{dataset}_processed_test.csv")

/tmp/ipykernel_8666/727029391.py:1: DtypeWarning: Columns (21,23) have mixed types. Specify dtype option on import or set low_memory=False.
  tab_train = pd.read_csv(f"datasets/processed/{dataset}_processed_train.csv")


In [7]:
if dataset == "BPIC15_4_f2":
    with open("dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]
elif dataset.startswith("BPIC15"):
    with open("dataset_features.json", 'r') as file:
        dataset_info = json.load(file)["BPIC15_common"]
else:
    with open("dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]

In [8]:
dataset_info

{'categorical': ['Responsible_actor',
  'CaseID',
  'Activity',
  'monitoringResource',
  'question',
  'Resource',
  'Aanleg (Uitvoeren werk of werkzaamheid)',
  'Bouw',
  'Brandveilig gebruik (melding)',
  'Brandveilig gebruik (vergunning)',
  'Flora en Fauna',
  'Gebiedsbescherming',
  'Handelen in strijd met regels RO',
  'Inrit/Uitweg',
  'Integraal',
  'Kap',
  'Milieu (melding)',
  'Milieu (neutraal wijziging)',
  'Milieu (omgevingsvergunning beperkte milieutoets)',
  'Milieu (vergunning)',
  'Monument',
  'Reclame',
  'Sloop'],
 'numerical': ['SUMleges',
  'duration',
  'month',
  'weekday',
  'hour',
  'remtime',
  'elapsed',
  'time:timestamp']}

In [9]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [10]:
'''
for k in categorical_columns:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")
''' 

for k in categorical_columns:
    tab_all[k] = tab_all[k].astype(str)
    tab_train[k] = tab_train[k].astype(str)
    tab_valid[k] = tab_valid[k].astype(str)
    tab_test[k] = tab_test[k].astype(str)

### Prepare the graphs

In [11]:
import sklearn.preprocessing

from typing import List

In [12]:
def get_case_ids(tab):
    return list(tab["CaseID"].unique())

In [13]:
from torch import tensor, max, int64, float32
from torch_geometric.data import HeteroData

In [14]:
def get_one_hot_encoder(dataset: pd.DataFrame, key: str):
    datas = dataset[key].unique()
    datas = datas.reshape([len(datas), 1])
    onehot = sklearn.preprocessing.OneHotEncoder()
    onehot.fit(datas)
    return onehot

In [15]:
def get_one_hot_encodings(
    onehot, datas: pd.Series
):
    return onehot.transform(datas.reshape(-1, 1)).toarray()

In [16]:
def get_node_features(dataset: pd.DataFrame, trace: pd.DataFrame, cat_features, real_features) -> dict:
 

    res = {}

    for key in trace:
        values = trace[key].values
        if key in cat_features:
            onehot_encoder = get_one_hot_encoder(dataset, key)
            try:
                res[key] = tensor(
                    get_one_hot_encodings(onehot_encoder, values),
                    dtype=float32,
                    requires_grad=True
                )
            except ValueError:
                print(key)
                print(values)
        if key in real_features:
            res[key] = tensor(values,  dtype=float32,requires_grad=True)
            res[key] = res[key].reshape(res[key].shape[0], 1)
        
    

    return res


In [17]:


def compute_edges_indexs(node_features: dict, prefix_len):
    res = {}
    keys = node_features.keys()
    
    indexes = [[i, i + 1] for i in range(prefix_len-1)]
   
    for k in keys:
        if len(node_features[k]) != 1:
            if k == "Activity":
                res[(k, "followed_by", k)] = indexes
                for k2 in keys:
                    if k2 != k:
                        if len(node_features[k2]) == 1:
                            res[(k, "related_to", k2)] = [
                                [i, 0] for i in range(prefix_len)
                            ]
                        else:
                            res[(k, "related_to", k2)] = [
                                [i, i] for i in range(prefix_len)
                            ]
            else:
                res[(k, "related_to", k)] = indexes

    return res

In [18]:



def build_prefixes_graph_from_trace(dataset, trace, cat_features, real_features, prefix_length):
    X = []  # graphs
   
    
    
    node_features = get_node_features(dataset, trace, cat_features, real_features)
    
    
    
    
    G = HeteroData()
        
        
        
    for k in node_features:
        if k != "case:label":
            G[k].x = node_features[k][:prefix_length]


    edges_indexes = compute_edges_indexs(node_features, prefix_length)

    


    for k in edges_indexes:
        ce = [[], []]
        for i in range(len(edges_indexes[k])):
            ce[0].append(edges_indexes[k][i][0])
            ce[1].append(edges_indexes[k][i][1])
        edges_indexes[k] = ce

    for k in edges_indexes:
        G[k].edge_index = tensor(edges_indexes[k], dtype=int64)


    ## Get the label of the trace
    label_value = trace["remaining_time"].values[prefix_length -1]
    G.y = tensor([label_value], dtype=float32)
    
        
    X.append(G)
    
    return X

## Create the datasets

In [19]:
case_train_ids = get_case_ids(tab_train)
case_valid_ids = get_case_ids(tab_valid)
case_test_ids = get_case_ids(tab_test)

In [20]:
print(len(case_train_ids))
print(len(case_valid_ids))
print(len(case_test_ids))

672
168
211


In [21]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)

In [22]:
trace = (
        tab_train.query(f"CaseID == '{case_train_ids[0]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )
trace 

,Responsible_actor,SUMleges,Aanleg (Uitvoeren werk of werkzaamheid),Bouw,Brandveilig gebruik (melding),Brandveilig gebruik (vergunning),Flora en Fauna,Gebiedsbescherming,Handelen in strijd met regels RO,Inrit/Uitweg,Integraal,Kap,Milieu (melding),Milieu (neutraal wijziging),Milieu (omgevingsvergunning beperkte milieutoets),Milieu (vergunning),Monument,Reclame,Sloop,Activity,monitoringResource,question,Resource,time:timestamp,duration,month,weekday,hour,remtime,elapsed,remaining_time
0,560600,0.0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_180\\complete,560600,EMPTY,560429,1.300915e+09,0.016667,3,3,1,0.0,14750458.0,0.0
1,560600,0.0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_150\\complete,560600,EMPTY,560429,1.300915e+09,0.000000,3,3,1,1.0,14750457.0,1.0
2,560600,0.0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_110\\complete,560600,False,560429,1.300915e+09,159380.950000,3,3,1,1.0,14750457.0,1.0
3,560600,0.0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_090_2\\complete,560600,EMPTY,560600,1.291352e+09,528.283333,12,4,9,9562858.0,5187600.0,9562858.0
4,560600,0.0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_090_1\\complete,560600,EMPTY,560600,1.291321e+09,100.866667,12,4,0,9594555.0,5155903.0,9594555.0
5,560600,0.0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_080\\complete,560600,42,560600,1.291315e+09,1.183333,12,3,22,9600607.0,5149851.0,9600607.0
6,560600,0.0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_070_2\\complete,560600,EMPTY,560600,1.291315e+09,0.433333,12,3,22,9600678.0,5149780.0,9600678.0
7,560600,0.0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_060\\complete,560600,True,560600,1.291315e+09,0.000000,12,3,22,9600704.0,5149754.0,9600704.0
8,560600,0.0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_070_1\\complete,560600,EMPTY,560600,1.291315e+09,0.100000,12,3,22,9600704.0,5149754.0,9600704.0
9,560600,0.0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,08_AWB45_030\\complete,560600,False,560600,1.291315e+09,0.400000,12,3,22,9600710.0,5149748.0,9600710.0


In [23]:
min_len = tab_all.groupby("CaseID").size().min()
max_len = tab_all.groupby("CaseID").size().max()
print("Minimum trace length:", min_len)
print("Maximum trace length:", max_len)

Minimum trace length: 5
Maximum trace length: 134


In [24]:
import pickle
from tqdm.notebook import tqdm

In [25]:
PREFIX_LENGTH = 4

In [26]:
print("Preparing training dataset...")

X_train = []


for i in tqdm(range(len(case_train_ids))):
    trace = (
        tab_train.query(f"CaseID == '{case_train_ids[i]}'")
        .reset_index(drop=True)
        .drop(columns="CaseID")
    )

    if len(trace) >= PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH,
        )
        for j in range(len(graphs)):
            X_train.append(graphs[j])

Preparing training dataset...


  0%|          | 0/672 [00:00<?, ?it/s]

In [27]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "wb") as f:
    pickle.dump(X_train, f)

In [28]:
print("Preparing validation dataset...")

X_valid = []


for i in tqdm(range(len(case_valid_ids))):
    trace = (
        tab_valid.query(f"CaseID == '{case_valid_ids[i]}'")
        .reset_index(drop=True)
        .drop(columns="CaseID")
    )
    if len(trace) >= PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH
        )
        for j in range(len(graphs)):
            X_valid.append(graphs[j])

Preparing validation dataset...


  0%|          | 0/168 [00:00<?, ?it/s]

In [29]:
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "wb") as f:
    pickle.dump(X_valid, f)

In [30]:
#comment
max_len_test = tab_test.groupby("CaseID").size().max()
MaxPrefix = min(20, max_len_test-1)
print("MaxPrefix: ", MaxPrefix)

MaxPrefix:  20


In [31]:
X_tests={}

for L in range(1,MaxPrefix+1):
    print(f"Preparing test dataset {L}...")
    X_test_L = []
    
    for i in tqdm(range(len(case_test_ids))):
        trace = (
            tab_test.query(f"CaseID == '{case_test_ids[i]}'")
            .reset_index()
            .drop(columns="index")
            .drop(columns="CaseID")
        )
        
        if len(trace) > L:  #generate_prefix_data in experiments/DatasetManager confirms >=
            graphs = build_prefixes_graph_from_trace(
                dataset=tab_all,
                trace=trace,
                cat_features=categorical_columns,
                real_features=real_value_columns,
                prefix_length=L
            )
            X_test_L.extend(graphs)
        
    X_tests[L] = X_test_L
        

Preparing test dataset 1...


  0%|          | 0/211 [00:00<?, ?it/s]

Preparing test dataset 2...


  0%|          | 0/211 [00:00<?, ?it/s]

Preparing test dataset 3...


...

  0%|          | 0/211 [00:00<?, ?it/s]

Preparing test dataset 19...


  0%|          | 0/211 [00:00<?, ?it/s]

Preparing test dataset 20...


  0%|          | 0/211 [00:00<?, ?it/s]

In [32]:
#with open(data_dir_graphs + dataset + "_TEST_repair.pkl", "wb") as f:
#    pickle.dump(X_test, f)


for L, X_test_L in X_tests.items():
    fname   = f"{dataset}_TEST{L}_repair.pkl"
    outpath = os.path.join(data_dir_graphs, fname)
    with open(outpath, "wb") as f:
        pickle.dump(X_test_L, f)
    print(f"Saved {len(X_test_L)} graphs for prefix {L} to {outpath}")

Saved 211 graphs for prefix 1 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2015_5_TEST1_repair.pkl
Saved 211 graphs for prefix 2 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2015_5_TEST2_repair.pkl
Saved 211 graphs for prefix 3 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2015_5_TEST3_repair.pkl
Saved 211 graphs for prefix 4 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2015_5_TEST4_repair.pkl
Saved 211 graphs for prefix 5 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2015_5_TEST5_repair.pkl
Saved 211 graphs for prefix 6 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2015_5_TEST6_repair.pkl
Saved 211 graphs for prefix 7 to /home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/bpic2015_5_TEST7_rep